In [1]:
from transformers import AutoModelForSequenceClassification, AutoConfig, AutoTokenizer, AdamW
import json
import torch
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import gc
import numpy as np
from scipy.stats import spearmanr
import openai
import time
import requests
import re
import pickle
from scipy.stats import spearmanr


openai.api_key = "YOUR_OPENAI_KEY"
device = torch.device("cuda:1")

np.random.seed(42)

# stage 0 score model 
optimize_stage = 2
complexity_model_name = f'../checkpoints/deberta-v3-large/alpaca-train-complexity-2k/stage_{optimize_stage}/rank_net-1024-ylabel'
quality_model_name = f'../checkpoints/deberta-v3-large/alpaca-train-quality-2k/_stage_{optimize_stage}_deberta-v3-large/rank_net-2048-ylabel'
tokenizer_path = '../pre_trained_model/deberta-v3-large'
data_path = '../data/deita_sota_pool/model_metric/_stage_0/qck_llama2-13b-hf_deita_sota_pool_305263_token_4096.json'
data_name = 'deita_sota_pool_305263_token_4096'
model_name = quality_model_name

print(data_name)
def _load_data(data_path: str) -> None:
    """
    Load data from data_path.
    data_path: str - path to json data file.
    """
    try:
        with open(data_path, "r") as f:
            data = json.load(f)
    except json.JSONDecodeError:
        with open(data_path, "r") as f:
            data = [json.loads(line) for line in f]
    return data
def read_pkl(fname):
    with open(fname, 'rb') as fo:
        pkl_data = pickle.load(fo, encoding='bytes')
    return pkl_data
def write_pkl(pkl_data, fname):
    fo = open(fname, 'wb')
    pickle.dump(pkl_data, fo)
    print("pkl_file write over!")

/home/pclv100s-1/.conda/envs/llama_factory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


deita_sota_pool_305263_token_4096


In [2]:
quality_data = _load_data(f'../data/RankGPT/stage_{optimize_stage}/alpaca-train-quality-2k.json')
quality_data[0]

{'query': ['Describe a method to improve engagement during online classes.',
  'Describe a method to improve engagement during online classes.',
  'Describe a method to improve engagement during online classes.',
  'Describe a method to improve engagement during online classes.',
  'Describe a method to improve engagement during online classes.'],
 'query_id': '',
 'positive_passages': '',
 'retrieved_passages': [{'instruction': 'Describe a method to improve engagement during online classes.',
   'output': 'To improve engagement during online classes, it is important to use a variety of teaching techniques. Interactive sessions like debates, group work, and small group activities can be particularly effective in keeping students engaged. Encouraging students to read and watch relevant materials and participate in discussion boards can also help to increase engagement. Regular assignments and assessments can keep students motivated, while rewarding and recognizing actively engaged stude

In [3]:
print(data_path)
data = _load_data(data_path)
print(data[0])

../data/deita_sota_pool/model_metric/_stage_0/qck_llama2-13b-hf_deita_sota_pool_305263_token_4096.json
{'id': 'LbHON3Q', 'source': 'ShareGPT', 'conversations': [{'from': 'human', 'value': 'People may now think of queries and replies in novel and creative ways thanks to new AI text systems. What are some crucial inquiries we may pose to these text-based AIs that would yield original and admirable insights into humanity? Make six specific questions that don’t contain the phrase “AI.”'}, {'from': 'gpt', 'value': '1. What is the meaning of life and how does it differ from individual to individual?\n2. How does our understanding of the world shape our beliefs and values?\n3. What are the most important qualities that make us human?\n4. How do our emotions influence our decision-making processes and behavior?\n5. What is the relationship between our thoughts and actions, and how does it impact our daily lives?\n6. Can humans find happiness and fulfillment by pursuing material wealth, or is t

In [4]:
model_name

'../checkpoints/deberta-v3-large/alpaca-train-quality-2k/_stage_2_deberta-v3-large/rank_net-2048-ylabel'

In [5]:
'''
batch complexity single 样本评分 
'''
import json
test_index = 1000


config = AutoConfig.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained('../pre_trained_model/deberta-v3-large', use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)


model.enable_input_require_grads()
model.gradient_checkpointing_enable()



quality_data = _load_data('../data/RankGPT/stage_1/alpaca-train-quality-2k.json')

query = quality_data[test_index]['query']
response = [quality_data[test_index]['retrieved_passages'][i]['text']  for i in range(len(quality_data[test_index]['retrieved_passages']))]


batch = tokenizer(query, response, padding=True, truncation=True, return_tensors="pt", max_length=2048)

out = model(**batch)
out.logits.squeeze().tolist()

/home/pclv100s-1/.conda/envs/llama_factory/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[2.928514003753662,
 0.5009137392044067,
 0.06798651069402695,
 -0.20347687602043152,
 -2.4018406867980957]

In [6]:
query

['Generate a code in Python to print "Hello World".',
 'are vegetables healthy?',
 'What are some of the main factors that determine the range and frequency of tides in different parts of the world?',
 "How can I modify the existing C# code to not only randomly select from five pre-written headlines, but also take into consideration the reader's geographic location and cultural background to generate a thought-provoking and attention-grabbing headline that highlights the cognitive, social, and economic benefits of becoming proficient in a second language? The headline should also incorporate recent studies and statistics on the increasing demand for multilingual individuals in the global job market, while maintaining a culturally sensitive and inclusive tone. Can you assist me in creating a headline that effectively communicates the nuanced and multifaceted nature of second language acquisition, while appealing to a diverse audience?",
 'Think back to a time in your career when you enc

In [22]:
device

device(type='cuda', index=1)

In [26]:
'''
single quality 样本评分 
'''
device='cuda:4'

config = AutoConfig.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
# model = model.cuda()
mdoel = model.to(device)
tokenizer = AutoTokenizer.from_pretrained('../pre_trained_model/deberta-v3-large', use_fast=True)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

quality_convs_score = []
for index,item in tqdm(enumerate(data[128501: ])):
    human_conversations = [conv['value'] for conv in item['conversations'][::2]]  # 偶数索引
    gpt_conversations   = [conv['value'] for conv in item['conversations'][1::2]] # 奇数索引
    
    if len(human_conversations) == len(gpt_conversations):
        conv_score = []
        for i in range(len(human_conversations)):
            batch = tokenizer(human_conversations[i], gpt_conversations[i], padding=True, truncation=True, return_tensors="pt", max_length=2048)
            # batch = {key: value.to(torch.device("cuda")) for key, value in batch.items()}
            batch = {key: value.to(device) for key, value in batch.items()}
            try:
                out = model(**batch)
            except:
                print('GC:\t', index)
                gc.collect()
                torch.cuda.empty_cache()                
                out = model(**batch)
            conv_score.append(out.logits.squeeze().tolist()) # out.logits.squeeze().tolist()=[2.9446518421173096, 10.687034606933594, 9.427430152893066]

        quality_convs_score.append(conv_score)
    else:
        print('error line:\t', index)

58003it [2:13:33,  6.43it/s]

: 

In [17]:
batch['input_ids']

tensor([[   1, 2433,  262,  ..., 1074,  260,    2]], device='cuda:1')

In [4]:

len(quality_convs_score)

NameError: name 'quality_convs_score' is not defined

In [25]:
write_pkl(quality_convs_score, './stage_0_101637_128500.pkl')

pkl_file write over!


In [ ]:
quality_model_name + f'/quality_stage_{optimize_stage}_{data_name}.pkl'

'../checkpoints/deberta-v3-base/alpaca-train-quality-2k/_stage_5/rank_net-2048-ylabel/quality_stage_5_deita_sota_pool_305263_token_4096.pkl'

In [ ]:
write_pkl(quality_convs_score, quality_model_name + f'/quality_stage_{optimize_stage}_{data_name}.pkl')

pkl_file write over!


In [ ]:
quality_convs_score[105]

[-0.9711354970932007,
 -3.0280921459198,
 3.835376024246216,
 -0.7333109378814697,
 -3.791534662246704,
 3.1756770610809326,
 -4.844959735870361]

In [ ]:
complexity_convs_score = read_pkl(f'{complexity_model_name}/complexity_stage_{optimize_stage}_{data_name}.pkl')

In [ ]:
add_complexity_quality_scores = []
for index,item in enumerate(data):
    item['complexity_scores'] = complexity_convs_score[index]
    item['quality_scores'] = quality_convs_score[index]
    add_complexity_quality_scores.append(item)



add_complexity_quality_scores_filename = '/'.join(data_path.split('/')[:-1]) + '/' + f'complexity_quality_stage_{optimize_stage}_{data_name}.json'
with open(add_complexity_quality_scores_filename, 'w') as f:
    json.dump(add_complexity_quality_scores, f)

In [ ]:
'''
Distillation Refine
'''


def _load_data(data_path: str) -> None:
    
    """
    Load data from data_path.
    
    data_path: str - path to json data file.
    """
    
    try:
        with open(data_path, "r") as f:
            data = json.load(f)
    except json.JSONDecodeError:
        with open(data_path, "r") as f:
            data = [json.loads(line) for line in f]
    
    return data

def random_selection_indices(data_list, num_bins=10, num_samples=3):
    # 计算直方图
    counts, bins = np.histogram(data_list, bins=num_bins)
    # print(f'counts:{counts}')
    # print(f'bins:{bins}')
    # 根据直方图划分的bin边界
    bin_edges = bins.tolist()
    # print(f'bin_edges:{bin_edges}')
    # 存储每个bin的索引
    bin_indices = [[] for _ in range(num_bins)]

    # 根据数值将元素索引添加到对应的bin
    for i, num in enumerate(data_list):
        for j in range(len(bin_edges) - 1):
            if bin_edges[j] <= num < bin_edges[j + 1]:
                bin_indices[j].append(i)
                break


    selected_indices = []
    for indices in bin_indices:
        if len(indices) >= num_samples:
            selected_indices.append(np.random.choice(indices, size=num_samples, replace=False).tolist())
        else:
            selected_indices.append(indices)

    return selected_indices, counts, bin_edges

# random_selection_indices(data_list=[i for i in range(10)] , num_bins=10, num_samples=3)

complexity_quality_data = _load_data(add_complexity_quality_scores_filename)
complexity_bins = []
quality_bins    = []
for index,item in enumerate(complexity_quality_data):
    complexity_bins.append(item['complexity_scores'][0])
    quality_bins.append(item['quality_scores'][0])

# 分数从小到大排列
quality_check_indices, quality_counts, quality_bin_edges = random_selection_indices(data_list=quality_bins, num_bins=5, num_samples=100)
complexity_check_indices, complexity_counts, complexity_bin_edges = random_selection_indices(data_list=complexity_bins, num_bins=5, num_samples=100)

print(f'max(quality_bins):{max(quality_bins)}, min(quality_bins):{min(quality_bins)},\nmax(complexity_bins):{max(complexity_bins)}, min(complexity_bins):{min(complexity_bins)}')

max(quality_bins):7.880617141723633, min(quality_bins):-8.374730110168457,
max(complexity_bins):30.9440860748291, min(complexity_bins):-25.298297882080078


In [ ]:
rank_quality_base_instruction = """Rank the following responses provided by different AI assistants to the user's question according to the quality of their responses.
Your evaluation should consider factors such as helpfulness, relevance, accuracy, depth, creativity, and level of detail of the response.
The following are {num} QUESTION and RESPONSE pairs, each indicated by number identifier [].
You can rank them based on their relevance to the question:

{question_response_pairs}

You will rank the {num} QUESTION and RESPONSE pairs above based on their relevance to the question.
The responses will be listed in descending order using identifiers, and the most relevant response should be listed first, and the output format should be [] > [] > etc, e.g., [1] > [2] > etc.
The ranking results of the {num} QUESTION and RESPONSE pairs (only identifiers) is:
"""


def createRankQualityPrompt(question_response_pairs, num, chat_mode=False):
    if not chat_mode:
        
        prompt = rank_quality_base_instruction.format(question_response_pairs=question_response_pairs, num=num)
        
        history=[{"role": "system", "content": "You are RankGPT, an intelligent assistant that can rank responses based on their questions."}, {"role": "user", "content": prompt}]
    
    else:
        history=[
            {"role": "system", "content": "You are RankGPT, an intelligent assistant that can rank responses based on their questions."},
            {"role": "user", "content": "I will provide you with {num} responses, each indicated by number identifier []. Rank them based on their relevance to question: {question}.\nYour evaluation should consider factors such as helpfulness, relevance, accuracy, depth, creativity, and level of detail of the response.".format(num, question)},
            {"role": "assistant", "content": "Okay, please provide the responses."}
        ]

        user_assistant_history = []
        for index,item in enumerate(responses):
            user_assistant_history.append(
               {"role": "user", "content": item}
            )
            user_assistant_history.append(
               {"role": "assistant", "content": "Received responses [{}]".format(index+1)}
            )
        history = history + user_assistant_history + [{
            "role": "assistant", "content": "Search question: {question}.\nRank the {num} responses above based on their relevance to the search question. The responses should be listed in descending order using identifiers, and the most relevant response should be listed first, and the output format should be [] > [], e.g., [1] > [2]. Only return the ranking results, do not say any word or explain.".format(question=question, num=num)
        }]
        
    return history, prompt




rank_complexity_base_instruction = """Ranking the following instructions provided by different users according to the instruction-following difficulty and complexity.
The following are {num} instructions, each indicated by number identifier [].
The user's instructions are: 

{instructions}

You will rank the {num} instructions above based on the instruction-following difficulty and complexity.
The instructions will be listed in descending order using identifiers, and the most relevant response should be listed first, and the output format should be [] > [] > etc, e.g., [1] > [2] > etc.
The ranking results of the {num} responses (only identifiers) is:
"""

def createRankComplexityPrompt(instructions, num, chat_mode=False):
    if not chat_mode:
        
        prompt = rank_complexity_base_instruction.format(instructions=instructions, num=num)
        
        history=[{"role": "system", "content": "You are RankGPT, an intelligent assistant that can rank instructions based on their instruction-following difficulty and complexity."}, {"role": "user", "content": prompt}]
    
    else:
        history=[
            {"role": "system", "content": "You are RankGPT, an intelligent assistant that can rank instructions based on the instruction-following difficulty and complexity."},
            {"role": "user", "content": "I will provide you with {num} instructions, each indicated by number identifier []. Rank them based on their instruction-following difficulty and complexity.".format(num)},
            {"role": "assistant", "content": "Okay, please provide the instructions."}
        ]

        user_assistant_history = []
        for index,item in enumerate(instructions):
            user_assistant_history.append(
               {"role": "user", "content": item}
            )
            user_assistant_history.append(
               {"role": "assistant", "content": "Received instructions [{}]".format(index+1)}
            )
        history = history + user_assistant_history + [{
            "role": "assistant", "content": "Rank the {num} instructions above based on their instruction-following difficulty and complexity. The instructions should be listed in descending order using identifiers, and the most complex and difficult instruction should be listed first, and the output format should be [] > [], e.g., [1] > [2]. Only return the ranking results, do not say any word or explain.".format(num=num)
        }]
        
    return history, prompt

In [ ]:
quality_check_indices = np.array(quality_check_indices).T
complexity_check_indices = np.array(complexity_check_indices).T

quality_history_list = []
quality_scores_list = []
quality_prompt_list = []
for index_list in quality_check_indices:
    q_a_list = []
    scores = []
    for i,index in enumerate(index_list):
        q = add_complexity_quality_scores[index]['conversations'][0]['value'] # q
        a = add_complexity_quality_scores[index]['conversations'][1]['value'] # a
        scores.append(add_complexity_quality_scores[index]['quality_scores'][0])
        q_a_list.append('['+str(int(i+1))+'] ' + f'QUESTION: {q}\n\nRESPONSE: {a}')
    question_response_pairs='\n'.join(q_a_list)
    history, prompt = createRankQualityPrompt(question_response_pairs=question_response_pairs, num=len(q_a_list))
    quality_prompt_list.append(prompt)
    quality_history_list.append(history)
    quality_scores_list.append(scores)

complexity_history_list = []
complexity_scores_list = []
complexity_prompt_list = []
for index_list in complexity_check_indices:
    q_list = []
    scores = []
    for i,index in enumerate(index_list):
        q = add_complexity_quality_scores[index]['conversations'][0]['value'] # q
        scores.append(add_complexity_quality_scores[index]['complexity_scores'][0])
        q_list.append('['+str(int(i+1))+'] ' + f'{q}')
    instructions='\n'.join(q_list)
    history, prompt = createRankComplexityPrompt(instructions=instructions, num=len(q_list))
    complexity_prompt_list.append(prompt)
    complexity_history_list.append(history)
    complexity_scores_list.append(scores)

# sorted(enumerate([1,2,3,4]), key=lambda x: x[1], reverse=True)
# [(3, 4), (2, 3), (1, 2), (0, 1)] (index, value)

def score_model_rank(scores):
    print(f'scores:{scores}')
    sorted_indexes = [idx+1 for idx, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]
    print(f'sorted_indexes:{sorted_indexes}')

